# ДЗ 3.3: Фракталы как вычислительный эксперимент  

---

## Введение и теоретические основы комплексной динамики

Рассматривается классическое семейство квадратичных комплексных отображений:

$$z_{n+1} = f_c(z_n) = z_n^2 + c, \qquad z, c \in \mathbb{C}.$$

Это простейшее нелинейное полиномиальное отображение порождает богатое фазовое поведение, служащее фундаментальной моделью в теории динамических систем и детерминированного хаоса.

### Основные определения:
1. **Заполненное множество Джулиа $K_c$ (Filled-in Julia set):**
   Для фиксированного комплексного параметра $c$ множество начальных точек $z_0 \in \mathbb{C}$, чьи траектории (орбиты) при итерациях остаются ограниченными:
   $$K_c = \{ z_0 \in \mathbb{C} : \sup_{n \ge 0} |z_n| < \infty \}.$$
   Топологическая граница $J_c = \partial K_c$ называется **множеством Джулиа**.

2. **Множество Мандельброта $\mathcal{M}$ (Mandelbrot set):**
   Множество всех комплексных параметров $c \in \mathbb{C}$, для которых заполненое множество Джулиа $K_c$ является связным. По фундаментальной теореме Фату–Жюлиа (1918–1919 гг.), связность $K_c$ эквивалентна тому, что критическая орбита, стартующая из единственной критической точки полинома $z_0 = 0$ ($f_c'(0) = 0$), не убегает на бесконечность:
   $$\mathcal{M} = \{ c \in \mathbb{C} : \sup_{n \ge 0} |z_n| < \infty, \; z_0 = 0 \}.$$

3. **Критерий убегания (Escape Criterion):**
   Если на некотором шаге $|z_k| > R$, где $R = \max(2, |c|)$ (а для множества Мандельброта $R = 2$), то по неравенству треугольника:
   $$|z_{k+1}| = |z_k^2 + c| \ge |z_k|^2 - |c| > |z_k| \cdot R - |c| \ge 2|z_k| - |c| > |z_k| + (|z_k| - |c|) > |z_k|.$$
   Отсюда при $|z_k| > R$ норма монотонно и сверхэкспоненциально возрастает к бесконечности: $|z_n| \to \infty$. Следовательно, $R$ является строгим радиусом убегания (escape radius).

**Цели работы:**
-  Реализовать векторизованный алгоритм на комплексной сетке с булевой маской активных точек; построить множество Мандельброта в области $\operatorname{Re} c \in [-2, 1]$, $\operatorname{Im} c \in [-1.5, 1.5]$ и множества Джулиа для $c \in \{0, -1, -0.123 + 0.745i, 1\}$; сопоставить случай $c = 0$ с единичным диском.
-  Исследовать увеличение участка границы (zoom) при 4 сочетаниях разрешений сетки ($600 \times 600, 1200 \times 1200$) и $N_{\max}$ ($200, 800$); вывести и реализовать формулу непрерывной плавной окраски (smooth coloring); построить двухпанельную анимацию эволюции множества Джулиа при движении $c$ через границу множества Мандельброта; проанализировать критическую орбиту $z_0 = 0$ и ответить на вопрос о связности и влиянии растрового разрешения.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle
from IPython.display import HTML, Markdown, display

# Настройки оформления графиков
plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 10,
    'axes.grid': False,
    'axes.spines.top': True,
    'axes.spines.right': True,
    'animation.embed_limit': 60,
    'animation.frame_format': 'jpeg'
})

def print_table(headers, rows):
    def clean_cell(x):
        s = str(x)
        parts = s.split('$')
        for k in range(len(parts)):
            if k % 2 == 1:
                parts[k] = parts[k].replace('|', r'\vert ')
            else:
                parts[k] = parts[k].replace('|', '&#124;')
        return '$'.join(parts)
    line = lambda items: '| ' + ' | '.join(clean_cell(x) for x in items) + ' |'
    md = [line(headers), line(['---'] * len(headers))] + [line(row) for row in rows]
    display(Markdown('\n'.join(md)))


# Общая палитра; черный означает отсутствие выхода за проверенное число шагов.
escape_cmap = plt.get_cmap('twilight_shifted').copy()
escape_cmap.set_bad('black')

def escape_image(grid, max_iter):
    return np.ma.masked_equal(grid, max_iter + 1)


## 1. Векторизованный алгоритм Escape-Time с булевой маской

Для эффективного расчета фракталов на сетках высокого разрешения используется векторизация на NumPy с динамической булевой маской:
1. Создается 2D-сетка комплексных чисел $Z$ (или $C$).
2. Заводится булева маска `mask = np.ones(shape, dtype=bool)`, отмечающая точки, чья траектория еще не превысила радиус $R$.
3. На каждой итерации $n = 1 \dots N_{\max}$ шаг отображения $z \leftarrow z^2 + c$ вычисляется **только для активных точек**: `Z[mask] = Z[mask]**2 + ...`.
4. Для точек, у которых $|z| > R$, фиксируется номер итерации выхода $n$, после чего они исключаются из маски: `mask[escaped] = False`.
5. Точки, оставшиеся в маске после $N_{\max}$ итераций, помечаются значением `max_iter + 1`: выход за проверенное число шагов не обнаружен; принадлежность заполненному множеству этим не доказана.

Такой подход сокращает объем вычислений в 3–5 раз по сравнению с наивным полным матричным обновлением.


In [ ]:
def compute_mandelbrot(xmin=-2.0, xmax=1.0, ymin=-1.5, ymax=1.5,
                       width=600, height=600, max_iter=200, R=2.0, smooth=False):
    # Генерация сетки параметров C
    x = np.linspace(xmin, xmax, width)
    y = np.linspace(ymin, ymax, height)
    C = x[np.newaxis, :] + 1j * y[:, np.newaxis]
    
    Z = np.zeros_like(C)
    iters = np.full(C.shape, max_iter + 1, dtype=float if smooth else int)
    mask = np.ones(C.shape, dtype=bool)
    log_R = np.log(R)
    
    for n in range(1, max_iter + 1):
        Z[mask] = Z[mask]**2 + C[mask]
        escaped_now = np.abs(Z[mask]) > R
        
        if np.any(escaped_now):
            sub_indices = np.where(escaped_now)[0]
            curr_mask_indices = np.where(mask)
            r_idx = curr_mask_indices[0][sub_indices]
            c_idx = curr_mask_indices[1][sub_indices]
            
            if smooth:
                mod_z = np.abs(Z[r_idx, c_idx])
                # Непрерывная поправка потенциала
                nu = n + 1.0 - np.log2(np.log(mod_z) / log_R)
                iters[r_idx, c_idx] = nu
            else:
                iters[r_idx, c_idx] = n
                
            mask[r_idx, c_idx] = False
            
        if not np.any(mask):
            break
            
    return x, y, iters


def compute_julia(c, xmin=-1.5, xmax=1.5, ymin=-1.5, ymax=1.5,
                  width=600, height=600, max_iter=200, R=None, smooth=False):
    if R is None:
        R = max(2.0, float(np.abs(c)))
        
    x = np.linspace(xmin, xmax, width)
    y = np.linspace(ymin, ymax, height)
    Z = x[np.newaxis, :] + 1j * y[:, np.newaxis]
    
    iters = np.full(Z.shape, max_iter + 1, dtype=float if smooth else int)
    mask = np.ones(Z.shape, dtype=bool)
    log_R = np.log(R)
    
    for n in range(1, max_iter + 1):
        Z[mask] = Z[mask]**2 + c
        escaped_now = np.abs(Z[mask]) > R
        
        if np.any(escaped_now):
            sub_indices = np.where(escaped_now)[0]
            curr_mask_indices = np.where(mask)
            r_idx = curr_mask_indices[0][sub_indices]
            c_idx = curr_mask_indices[1][sub_indices]
            
            if smooth:
                mod_z = np.abs(Z[r_idx, c_idx])
                nu = n + 1.0 - np.log2(np.log(mod_z) / log_R)
                iters[r_idx, c_idx] = nu
            else:
                iters[r_idx, c_idx] = n
                
            mask[r_idx, c_idx] = False
            
        if not np.any(mask):
            break
            
    return x, y, iters


---
## 2. Множество Мандельброта и множества Джулиа

Построим:
1. **Множество Мандельброта** в области $\operatorname{Re} c \in [-2, 1]$, $\operatorname{Im} c \in [-1.5, 1.5]$ на сетке $600 \times 600$ при $N_{\max} = 200$.
2. **Четыре заполненных множества Джулиа** для параметров:
   * $c_1 = 0$ (гладкий единичный диск, граница — окружность $|z|=1$);
   * $c_2 = -1$ (базилика с бесконечным числом соединенных компонент);
   * $c_3 = -0.123 + 0.745i$ (дендритный «кролик» Дуади со спиралями);
   * $c_4 = 1$ (параметр вне множества Мандельброта, канторова пыль).

Цветовая палитра: точки, вышедшие за радиус $R$, окрашиваются по номеру первой итерации убегания (палитра `twilight_shifted`), а точки, не вышедшие за $N_{\max} = 200$ шагов, показаны черным. Это конечное приближение, а не доказанная принадлежность множеству. Для всех пяти карт используется одна шкала от 0 до 200.


In [ ]:
# 1. Расчет множества Мандельброта
x_m, y_m, m_grid = compute_mandelbrot(xmin=-2.0, xmax=1.0, ymin=-1.5, ymax=1.5,
                                      width=600, height=600, max_iter=200, R=2.0)

# Параметры Джулиа из задания
c_list = [
    0.0 + 0.0j,
    -1.0 + 0.0j,
    -0.123 + 0.745j,
    1.0 + 0.0j
]
c_names = [
    r'$c_1 = 0$',
    r'$c_2 = -1$',
    r'$c_3 = -0.123 + 0.745i$',
    r'$c_4 = 1$'
]
colors_pts = ['red', 'cyan', 'lime', 'magenta']

# Одинаковый номер выхода имеет одинаковый цвет на всех пяти картах.
norm_base = plt.Normalize(vmin=0, vmax=200)

# Построение карты Мандельброта с отметками параметров
fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
im_m = ax.imshow(escape_image(m_grid, 200), extent=[-2.0, 1.0, -1.5, 1.5], origin='lower',
                 cmap=escape_cmap, norm=norm_base, interpolation='nearest', aspect='equal')
ax.set_title(r'Множество Мандельброта $\mathcal{M}$ ($600 \times 600, \; N_{\max} = 200$)')
ax.set_xlabel(r'$\operatorname{Re}(c)$')
ax.set_ylabel(r'$\operatorname{Im}(c)$')

# Нанесение 4 точек
for c_val, c_lbl, c_col in zip(c_list, c_names, colors_pts):
    ax.plot(c_val.real, c_val.imag, 'o', color=c_col, ms=8, mec='black', mew=1.2,
            label=f'{c_lbl}')

ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
cbar = fig.colorbar(im_m, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label(r'Номер выхода $n$; черный: выход не обнаружен за 200 шагов')
plt.show()

# 2. Построение 4 множеств Джулиа
fig, axes = plt.subplots(2, 2, figsize=(11, 10), constrained_layout=True)

julia_grids = []
for ax, c_val, c_lbl, c_col in zip(axes.flat, c_list, c_names, colors_pts):
    R_val = max(2.0, float(np.abs(c_val)))
    xj, yj, j_grid = compute_julia(c_val, xmin=-1.6, xmax=1.6, ymin=-1.6, ymax=1.6,
                                   width=600, height=600, max_iter=200, R=R_val)
    julia_grids.append(j_grid)
    
    im_j = ax.imshow(escape_image(j_grid, 200), extent=[-1.6, 1.6, -1.6, 1.6], origin='lower',
                     cmap=escape_cmap, norm=norm_base, interpolation='nearest', aspect='equal')
    ax.set_title(f'Множество Джулиа для {c_lbl} (R = {R_val:.2f})')
    ax.set_xlabel(r'$\operatorname{Re}(z)$')
    ax.set_ylabel(r'$\operatorname{Im}(z)$')
    
    # Для c = 0 сопоставляем с единичным диском |z| <= 1 и окружностью |z| = 1
    if c_val == 0.0:
        circle = Circle((0, 0), 1.0, color='red', fill=False, ls='--', lw=1.8, label=r'Окружность $|z|=1$')
        ax.add_patch(circle)
        ax.legend(loc='upper right', fontsize=8)

fig.colorbar(im_j, ax=axes.ravel().tolist(), fraction=0.035, pad=0.03,
             label='Номер выхода n; черный: выход не обнаружен за 200 шагов')
fig.suptitle(r'Заполненные множества Джулиа $K_c$ ($600 \times 600, \; N_{\max}=200$)', fontsize=13)
plt.show()

# Сводная таблица параметров
print_table(['Параметр c', '|c|', 'Радиус выхода R', 'Положение относительно M', 'Топология множества Джулиа'], [
    [r'$c_1 = 0$', '0.00', '2.00', 'В центре главной кардиоиды', r'Гладкий единичный круг $|z| \le 1$'],
    [r'$c_2 = -1$', '1.00', '2.00', 'На вещественной оси (bulb периода 2)', 'Связное, бесконечное число перетяжек (базилика)'],
    [r'$c_3 = -0.123 + 0.745i$', '0.75', '2.00', 'Внутри лепестка периода 3 (связная область M)', 'Связное, фрактальные спирали (кролик Дуади)'],
    [r'$c_4 = 1$', '1.00', '2.00', 'Вне множества Мандельброта', 'Вполне несвязное канторово множество (пыль)']
])


### Аналитическое сопоставление при $c = 0$

При $c = 0$ комплексное отображение имеет тривиальный вид:
$$z_{n+1} = z_n^2 \implies z_n = z_0^{2^n} \implies |z_n| = |z_0|^{2^n}.$$

Анализ устойчивости орбит:
1. **$|z_0| < 1$:** $|z_n| \to 0$ при $n \to \infty$ — орбита стремится к устойчивой суперпритягивающей неподвижной точке $z = 0$.
2. **$|z_0| = 1$:** $|z_n| = 1$ для всех $n \ge 0$ — модуль орбиты сохраняется строго постоянным, точка бесконечно вращается по окружности.
3. **$|z_0| > 1$:** $|z_n| \to \infty$ — норма лавинообразно убегает на бесконечность.

Следовательно:
* Заполненное множество Джулиа есть **точный замкнутый единичный диск**:
  $$K_0 = \{ z \in \mathbb{C} : |z| \le 1 \}.$$
* Множество Джулиа (граница $K_0$) — **единичная окружность**:
  $$J_0 = \partial K_0 = \{ z \in \mathbb{C} : |z| = 1 \}.$$

Красная пунктирная линия на графике выше строго накладывается на границу черного диска, что подтверждает корректность численного алгоритма.



> **Вопрос:** *Что можно утверждать о точке, которая не вышла за радиус за конечное число шагов?*

**Ответ:**
1. **Строгое математическое утверждение:**
   О точке $z_0$, для которой $|z_n| \le R$ при всех $n \le N_{\max}$, можно утверждать **лишь то, что её орбита не покинула круг радиуса $R$ за проверенные $N_{\max}$ итераций**.
2. **Почему нельзя строго утверждать принадлежность множеству:**
   Вблизи границы фрактала $J_c$ (а также $\partial \mathcal{M}$) время до выхода может быть сколь угодно большим. Существуют точки, лежащие вне заполненного множества, орбиты которых покидают круг радиуса $R$ только через $N = 300, 1000$ или $10^5$ шагов. 
3. **Численный смысл:**
   В точной арифметике область $K_c^{(N)} = \{z_0: |z_n| \le R,\ 0 \le n \le N\}$ является **надмножеством** заполненного множества:
   $$K_c \subset K_c^{(N_{\max})}, \qquad K_c = \bigcap_{N=1}^\infty K_c^{(N)}.$$
   При увеличении $N$ эти области сужаются к $K_c$. Растровая картинка показывает только значения на выбранной сетке и дополнительно зависит от разрешения и ошибок округления. Ни 40, ни 64, ни любое другое конечное число шагов само по себе не доказывает ограниченность орбиты.


---
## 3. Исследование разрешения и $N_{\max}$ при увеличении (Zoom)

Выберем участок границы множества Мандельброта с богатой спиральной структурой — область долины морских коньков (Seahorse Valley):
$$\operatorname{Re} c \in [-0.755, -0.745], \qquad \operatorname{Im} c \in [0.095, 0.105].$$

Сравним четыре комбинации расчетной сетки и лимита итераций:
1. Сетка $600 \times 600$, $N_{\max} = 200$;
2. Сетка $1200 \times 1200$, $N_{\max} = 200$;
3. Сетка $600 \times 600$, $N_{\max} = 800$;
4. Сетка $1200 \times 1200$, $N_{\max} = 800$.

Все четыре изображения строятся с одной шкалой номера выхода от 0 до 800, без деления на $N_{\max}$. Черным показаны точки, не вышедшие за свой лимит; при 200 и 800 шагах смысл этой отметки различается.


In [ ]:
import time

zoom_box = dict(xmin=-0.755, xmax=-0.745, ymin=0.095, ymax=0.105)
combos = [
    (600, 200, '600x600, N=200'),
    (1200, 200, '1200x1200, N=200'),
    (600, 800, '600x600, N=800'),
    (1200, 800, '1200x1200, N=800')
]

zoom_results = []
zoom_times = []
interior_counts = []

for w, n_max, label in combos:
    t0 = time.time()
    _, _, grid_z = compute_mandelbrot(**zoom_box, width=w, height=w, max_iter=n_max, R=2.0)
    dt = time.time() - t0
    zoom_results.append((grid_z, n_max, label))
    zoom_times.append(dt)
    interior_counts.append(np.sum(grid_z == n_max + 1))

# Построение 4 панелей с общей шкалой абсолютного номера выхода
norm_zoom = plt.Normalize(vmin=0, vmax=800)
fig, axes = plt.subplots(2, 2, figsize=(11, 10), constrained_layout=True)

for ax, (grid_z, n_max, label), dt, count in zip(axes.flat, zoom_results, zoom_times, interior_counts):
    norm_grid = escape_image(grid_z, n_max)
    im = ax.imshow(norm_grid, extent=[zoom_box['xmin'], zoom_box['xmax'], zoom_box['ymin'], zoom_box['ymax']],
                   origin='lower', cmap=escape_cmap, norm=norm_zoom, interpolation='nearest', aspect='equal')
    ax.set_facecolor('black')  # Выход не обнаружен за n_max шагов
    ax.set_title(f'{label} (Время: {dt:.2f} с)')
    ax.set_xlabel(r'$\operatorname{Re}(c)$')
    ax.set_ylabel(r'$\operatorname{Im}(c)$')

fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.035, pad=0.03,
             label='Номер выхода n; черный: выход не обнаружен за N_max шагов')
fig.suptitle('Фрагмент долины морских коньков (Seahorse Valley) при различных сетках', fontsize=13)
plt.show()

# Сводная таблица параметров увеличения
print_table(['Разрешение', 'N_max', 'Время расчета (с)', 'Без выхода за N_max шагов', 'Доля точек без обнаруженного выхода'], [
    [f'{w}x{w}', n_max, f'{dt:.3f}', f'{count}', f'{count / (w*w) * 100:.2f}%']
    for (w, n_max, label), dt, count in zip(combos, zoom_times, interior_counts)
])


---
## 4. Непрерывная плавная окраска (Smooth Coloring)

При классической дискретной окраске цвет пикселя определяется целым числом $n$ (номер шага выхода за радиус $R$). Это создает искусственные ступенчатые полосы равного потенциала (color banding), маскирующие непрерывную структуру фрактала.

### Математический вывод формулы:
Вдали от множества динамика $z_{n+1} \approx z_n^2$ ведет себя как чистое возведение в квадрат:
$$|z_n| \approx |z_0|^{2^n} \implies \ln |z_n| \approx 2^n \ln |z_0| \implies \ln(\ln |z_n|) \approx n \ln 2 + \ln(\ln |z_0|).$$

Определим непрерывный индекс $\nu(z_0)$ такой, что при фиктивном непрерывном номере шага норма точно достигает радиуса убегания: $|z_\nu| = R$.  
Тогда:
$$\ln(\ln R) \approx \nu \ln 2 + \ln(\ln |z_0|) = n \ln 2 + \ln(\ln |z_0|) + \ln(\ln |z_n|) - n \ln 2 \dots$$
Вычитая члены, получаем формулу дробной поправки:
$$\nu = n + 1 - \frac{\ln\left(\frac{\ln |z_n|}{\ln R}\right)}{\ln 2} = n + 1 - \log_2 \left( \frac{\ln |z_n|}{\ln R} \right),$$
где $z_n$ — значение комплексного числа в момент первого пересечения $|z_n| > R$.

Дробная добавка компенсирует тот факт, что орбита «перелетает» порог $R$ на разную величину, и превращает ступенчатый целочисленный счетчик в гладкую дифференцируемую функцию координат.


In [ ]:
# Сравнение дискретной и плавной окраски одного фрагмента
_, _, discrete_grid = compute_mandelbrot(**zoom_box, width=800, height=800, max_iter=400, R=2.0, smooth=False)
_, _, smooth_grid   = compute_mandelbrot(**zoom_box, width=800, height=800, max_iter=400, R=2.0, smooth=True)

norm_smooth = plt.Normalize(vmin=0, vmax=400)
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), constrained_layout=True)

# 1. Дискретная окраска
im1 = axes[0].imshow(escape_image(discrete_grid, 400), extent=[zoom_box['xmin'], zoom_box['xmax'], zoom_box['ymin'], zoom_box['ymax']],
                     origin='lower', cmap=escape_cmap, norm=norm_smooth, interpolation='nearest', aspect='equal')
axes[0].set_title('Дискретная окраска (видны полосы / banding)')
axes[0].set_xlabel(r'$\operatorname{Re}(c)$')
axes[0].set_ylabel(r'$\operatorname{Im}(c)$')

# 2. Плавная окраска
im2 = axes[1].imshow(escape_image(smooth_grid, 400), extent=[zoom_box['xmin'], zoom_box['xmax'], zoom_box['ymin'], zoom_box['ymax']],
                     origin='lower', cmap=escape_cmap, norm=norm_smooth, interpolation='nearest', aspect='equal')
axes[1].set_title('Плавная непрерывная окраска (Continuous Potential)')
axes[1].set_xlabel(r'$\operatorname{Re}(c)$')
axes[1].set_ylabel(r'$\operatorname{Im}(c)$')

fig.colorbar(im2, ax=axes.tolist(), fraction=0.035, pad=0.03,
             label='Индекс выхода n / ν; черный: выход не обнаружен за 400 шагов')
fig.suptitle('Сравнение методов окраски границы множества Мандельброта', fontsize=13)
plt.show()


---
## 5. Анимация движения $c$ через границу Мандельброта

Построим двухпанельную анимацию изменения заполненного множества Джулиа $K_{c(t)}$ при движении параметра:
$$c(t) = -0.123 + i \cdot (0.55 + 0.50 t), \qquad t \in [0, 1].$$

В начале отрезка $c = -0.123 + 0.55i$ находится внутри главной кардиоиды, а при $t \approx 0.39$ параметр проходит вблизи центра лепестка периода 3 — области кролика Дуади. В конце отрезка $c = -0.123 + 1.05i$ находится вне $\mathcal{M}$.

Для кадра с индексом 40 ($t = 40/59$, $c \approx -0.123 + 0.888983i$) критическая орбита выходит за $R=2$ на 32-м шаге. Поэтому этот параметр уже вне $\mathcal{M}$, а $K_c$ вполне несвязно. Его вытянутые фрагменты могут выглядеть соединенными на растре. Канторова структура возникает сразу при выходе параметра из $\mathcal{M}$; при дальнейшем движении становятся заметнее промежутки между фрагментами. Этот кадр не определяет точное положение границы.

Слева показана траектория на карте Мандельброта, справа — приближение $K_c$. Обе панели и все кадры используют одну палитру и фиксированную шкалу плавного индекса от 0 до 150. Черный цвет означает, что выход за 150 шагов не обнаружен.


In [ ]:
# Параметры траектории параметра c(t)
n_anim_frames = 60
t_vals = np.linspace(0, 1, n_anim_frames)
c_traj = -0.123 + 1j * (0.55 + 0.50 * t_vals)

# Фоновая карта Мандельброта для левой панели
x_bg, y_bg, bg_m = compute_mandelbrot(xmin=-1.5, xmax=0.5, ymin=0.2, ymax=1.2,
                                      width=350, height=350, max_iter=150, smooth=True)

# Предварительный расчет кадров Джулиа
julia_frames = []
for c_curr in c_traj:
    _, _, j_frame = compute_julia(c_curr, xmin=-1.4, xmax=1.4, ymin=-1.4, ymax=1.4,
                                  width=350, height=350, max_iter=150, smooth=True)
    julia_frames.append(j_frame)

norm_anim = plt.Normalize(vmin=0, vmax=150)
fig, (ax_mand, ax_jul) = plt.subplots(1, 2, figsize=(11, 5.0), constrained_layout=True)

# Левая панель: Мандельброт и траектория
im_m_anim = ax_mand.imshow(escape_image(bg_m, 150), extent=[-1.5, 0.5, 0.2, 1.2], origin='lower',
                           cmap=escape_cmap, norm=norm_anim, interpolation='nearest', aspect='equal')
ax_mand.plot(c_traj.real, c_traj.imag, 'r--', lw=1.5, label='Траектория $c(t)$')
c_marker, = ax_mand.plot([], [], 'o', color='yellow', ms=8, mec='black', mew=1.5, label='Текущий $c$')
ax_mand.set_title('Карта параметров (Мандельброт)')
ax_mand.set_xlabel(r'$\operatorname{Re}(c)$')
ax_mand.set_ylabel(r'$\operatorname{Im}(c)$')
ax_mand.legend(loc='lower left', fontsize=8)

# Правая панель: Множество Джулиа
im_j_anim = ax_jul.imshow(escape_image(julia_frames[0], 150), extent=[-1.4, 1.4, -1.4, 1.4], origin='lower',
                          cmap=escape_cmap, norm=norm_anim, interpolation='nearest', aspect='equal')
ax_jul.set_facecolor('black')
title_jul = ax_jul.set_title('')
ax_jul.set_xlabel(r'$\operatorname{Re}(z)$')
ax_jul.set_ylabel(r'$\operatorname{Im}(z)$')

fig.colorbar(im_j_anim, ax=[ax_mand, ax_jul], fraction=0.025, pad=0.03,
             label='Индекс выхода ν; черный: нет выхода за 150 шагов')
heading_anim = fig.suptitle('')

def update_fractal_anim(frame_idx):
    c_now = c_traj[frame_idx]
    c_marker.set_data([c_now.real], [c_now.imag])
    
    im_j_anim.set_data(escape_image(julia_frames[frame_idx], 150))
    title_jul.set_text(fr'Множество Джулиа: $c = {c_now.real:.3f} + {c_now.imag:.3f}i$')
    heading_anim.set_text(f'Эволюция множества Джулиа при пересечении границы Мандельброта (кадр {frame_idx+1}/{n_anim_frames})')
    return c_marker, im_j_anim, title_jul, heading_anim

anim_fractal = FuncAnimation(fig, update_fractal_anim, frames=n_anim_frames, interval=70, blit=False)
plt.close(fig)

anim_html_fractal = anim_fractal.to_jshtml(fps=14, default_mode='once')
display(HTML(anim_html_fractal))


In [ ]:
# Критические орбиты для трех параметров из анимации.
sample_indices = [5, 40, 58]
sample_names = ['Внутри главной кардиоиды', 'Вне M, медленное убегание', 'Вне M, быстрое убегание']
colors_orbit = ['#1f77b4', '#ff7f0e', '#2ca02c']
orbit_limits = [40, 64, 200, 800]
n_steps_orbit = max(orbit_limits)
orbit_table_data = []

fig, ax = plt.subplots(figsize=(9, 4.4), constrained_layout=True)
for idx, label, col in zip(sample_indices, sample_names, colors_orbit):
    c_test = c_traj[idx]
    z_orbit = [0.0j]
    esc_step = None
    for step in range(1, n_steps_orbit + 1):
        z_next = z_orbit[-1]**2 + c_test
        z_orbit.append(z_next)
        if abs(z_next) > 2.0:
            esc_step = step
            break

    # Показываем только рассчитанные шаги; после пересечения R=2 убегание установлено.
    mod_orbit = np.abs(z_orbit)
    ax.plot(np.arange(len(mod_orbit)), mod_orbit, 'o-', color=col, lw=1.6, ms=4,
            label=f'{label} ($c = {c_test.real:.3f} + {c_test.imag:.6f}i$)')

    # Проверка принадлежности внутренности главной кардиоиды.
    x, y = c_test.real, c_test.imag
    q = (x - 0.25)**2 + y*y
    in_cardioid = q * (q + x - 0.25) < 0.25 * y*y
    if esc_step is not None:
        topo_desc = 'Вполне несвязное канторово множество: критическая орбита вышла за R=2'
    elif in_cardioid:
        topo_desc = 'Связное: параметр внутри главной кардиоиды (аналитический критерий)'
    else:
        topo_desc = 'По конечному числу итераций связность не установлена'

    checks = [f'Выход на шаге {esc_step}' if esc_step is not None and esc_step <= limit
              else f'При N={limit} выход не обнаружен' for limit in orbit_limits]
    orbit_table_data.append([label, f'{x:.3f} + {y:.6f}i', *checks, topo_desc])

ax.axhline(2.0, color='red', ls='--', lw=1.2, label='Порог убегания $R = 2$')
ax.set_yscale('log')
ax.set_ylim(1e-1, 6)
ax.set_xlim(0, 64)
ax.set_xlabel('Номер итерации $n$ (первые 64 шага)')
ax.set_ylabel(r'Модуль орбиты $|z_n|$')
ax.set_title(r'Критическая орбита $z_0 = 0$: расчет до выхода за $R=2$ или до 800 шагов')
ax.legend(loc='lower right', fontsize=8)
plt.show()
print_table(['Положение c', 'Параметр c', *[f'N={v}' for v in orbit_limits], 'Топология K_c'],
            orbit_table_data)

# При одном и том же c сравниваем конечные приближения за 40 и 64 шага.
c_compare = c_traj[40]
fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
norm_limits = plt.Normalize(vmin=0, vmax=64)
for ax, limit in zip(axes, [40, 64]):
    _, _, grid = compute_julia(c_compare, xmin=-1.4, xmax=1.4, ymin=-1.4, ymax=1.4,
                               width=600, height=600, max_iter=limit)
    im = ax.imshow(escape_image(grid, limit), extent=[-1.4, 1.4, -1.4, 1.4],
                   origin='lower', cmap=escape_cmap, norm=norm_limits,
                   interpolation='nearest', aspect='equal')
    ax.set_title(f'N_max = {limit}; без выхода: {np.count_nonzero(grid == limit + 1)} точек')
    ax.set_xlabel(r'$\operatorname{Re}(z)$')
    ax.set_ylabel(r'$\operatorname{Im}(z)$')
fig.colorbar(im, ax=axes.tolist(), fraction=0.03, pad=0.03,
             label='Номер выхода n; черный: выход не обнаружен за N_max шагов')
fig.suptitle(fr'$c = {c_compare.real:.3f} + {c_compare.imag:.6f}i$: влияние лимита итераций')
plt.show()



> **Вопрос:** *Как связаны ограниченность критической орбиты и связность множества Джулиа, и какие ограничения накладывает конечное разрешение картинки?*

**Ответ:**
Для $f_c(z)=z^2+c$ единственная критическая точка — $z=0$. Если ее орбита ограничена, то $c\in\mathcal{M}$, а $K_c$ и $J_c$ связны. Если критическая орбита убегает, то $c\notin\mathcal{M}$, а $K_c=J_c$ — вполне несвязное канторово множество. Отдельной стадии «уже вне $\mathcal{M}$, но еще связно» в этом семействе нет.

Для выбранных параметров выход обнаружен на шагах 32 и 8. Поэтому оба множества вполне несвязны, хотя первое сохраняет на картинке вытянутые скопления точек. Для внутреннего параметра одного отсутствия выхода за 800 шагов недостаточно. Здесь связность следует из принадлежности внутренности главной кардиоиды: при $c=x+iy$, $q=(x-1/4)^2+y^2$ выполняется $q(q+x-1/4)<y^2/4$. В этой области есть притягивающая неподвижная точка, и критическая орбита ограничена.

Конечные итерации и конечное разрешение дают разные ошибки. При малом $N_{\max}$ остаются точки, которые выйдут позже; сравнение 40 и 64 шагов показывает изменение черной области при неизменном параметре и сетке. Увеличение любого конечного лимита не превращает отсутствие выхода в доказательство ограниченности. При грубой сетке узкие промежутки могут исчезать, а тонкие части связного множества — пропускаться. Поэтому растровое изображение и измельчение сетки сами по себе не доказывают связность.


---
## 6. Выводы

1. **Векторизованный вычислительный эксперимент:**
   - Реализован эффективный алгоритм escape-time с булевой маской активных точек на комплексной сетке NumPy.
   - Построено множество Мандельброта в канонической области $\operatorname{Re} c \in [-2, 1]$, $\operatorname{Im} c \in [-1.5, 1.5]$ и четыре множества Джулиа для $c \in \{0, -1, -0.123+0.745i, 1\}$.
   - Для $c = 0$ получено совпадение в пределах разрешения сетки с единичным диском $|z| \le 1$ и единичной окружностью $|z| = 1$.
   - В точной арифметике область без выхода за конечное число шагов содержит заполненное множество; на сетке рассчитывается только ее растровое приближение.

2. **Влияние разрешения и непрерывная окраска:**
   - Исследовано увеличение участка границы в Seahorse Valley при 4 комбинациях сетки ($600 \times 600, 1200 \times 1200$) и $N_{\max}$ ($200, 800$). Показано, что повышение пространственного разрешения устраняет пикселизацию, а увеличение $N_{\max}$ отсекает ложные внутренние области.
   - Выведена и реализована формула плавной окраски $\nu = n + 1 - \log_2(\ln |z_n| / \ln R)$, сглаживающая ступенчатые полосы. Для сравниваемых панелей зафиксированы общие цветовые шкалы, точки без обнаруженного выхода выделены черным.

3. **Связность множеств Джулиа и критическая орбита:**
   - Создана и встроена двухпанельная анимация движения параметра $c(t)$ через границу множества Мандельброта с синхронным отображением множества Джулиа.
   - Для двух внешних параметров критическая орбита вышла за $R=2$ на шагах 32 и 8: оба множества вполне несвязны. Для внутреннего параметра связность обоснована принадлежностью главной кардиоиде, а не отсутствием выхода за конечное число шагов.
   - Сопоставлены лимиты 40, 64, 200 и 800 шагов. Отсутствие выхода за конечное число шагов не доказывает ограниченность орбиты; изображение дополнительно зависит от разрешения сетки.
